# Notebook 08 - Actividad Final: Asistente Documental Inteligente

## Objetivos
- Integrar sentimiento BERT, QA y resumen GPT.
- Procesar `documentos_qa.csv` y textos personalizados.
- Documentar configuracion, resultados y mejoras futuras.

## Introduccion
Proyecto integrador resuelto: un pipeline que analiza documentos, responde preguntas, estima tono y genera resumenes ejecutivos.

In [1]:
# Escribe tu codigo aqui
from pathlib import Path
from dataclasses import dataclass
from IPython.display import display
import pandas as pd
from transformers import pipeline, set_seed

set_seed(42)

RUTA_QA = Path("..") / "datasets" / "documentos_qa.csv"

## 1) Configuracion del pipeline

In [4]:
# Escribe tu codigo aqui
@dataclass
class ConfigAsistente:
    model_sentimientos: str = "pysentimiento/robertuito-sentiment-analysis"
    model_qa: str = "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"
    model_generacion_texto: str = "datificate/gpt2-small-spanish"
    max_new_tokens: int = 60
    temperature: float = 0.6

cfg = ConfigAsistente()
print(f"Configuration: {cfg}")

Configuration: ConfigAsistente(model_sentimientos='pysentimiento/robertuito-sentiment-analysis', model_qa='mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es', model_generacion_texto='datificate/gpt2-small-spanish', max_new_tokens=60, temperature=0.6)


## 2) Inicializar componentes

In [5]:
# Escribe tu codigo aqui
sentimientos_pipeline = pipeline(
    'sentiment-analysis',
    model=cfg.model_sentimientos
)
qa_pipeline = pipeline(
    'question-answering',
    model=cfg.model_qa
)
gen_pipeline = pipeline(
    'text-generation',
    model=cfg.model_generacion_texto
)
print("Pipelines loaded successfully.")

Device set to use cpu
Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu


Pipelines loaded successfully.


## 3) Funciones del asistente

In [6]:
# Escribe tu codigo aqui
def analizar_sentimientos(texto: str) -> dict:
    out = sentimientos_pipeline(texto)[0]
    return {'label': out['label'], 'score': round(out['score'], 4)}

def responder_pregunta(contexto: str, pregunta: str) -> dict:
    out = qa_pipeline(
        question=pregunta,
        context=contexto
    )
    return {'answer': out['answer'], 'score': round(out['score'], 4)}

def resumir_documento(texto: str) -> str:
    prompt = f"Resume el siguiente texto en una frase: {texto}"
    salida = gen_pipeline(
        prompt,
        max_new_tokens=cfg.max_new_tokens,
        do_sample=True,
        temperature=cfg.temperature
    )
    return salida[0]['generated_text']

## 4) Procesar documentos_qa.csv

In [7]:
# Escribe tu codigo aqui
df = pd.read_csv(RUTA_QA)
display(df.head(3))

,contexto,pregunta,respuesta
0,Python fue creado por Guido van Rossum y publi...,¿Quién creó Python?,Guido van Rossum
1,Python fue creado por Guido van Rossum y publi...,¿En qué año se publicó Python?,1991
2,Los Transformers fueron introducidos en el pap...,¿Qué empresa introdujo los Transformers?,Google


In [ ]:
df = pd.read_csv(RUTA_QA)
resultados = []

for _, row in df.iterrows():
    contexto = row['contexto']
    pregunta = row['pregunta']
    qa = responder_pregunta(contexto, pregunta)
    sent = analizar_sentimientos(contexto)
    resumen = resumir_documento(contexto)
    resultados.append({
        'pregunta': pregunta,
        'respuesta_esperada': row['respuesta'],
        'respuesta_modelo': qa['answer'],
        'qa_score': qa['score'],
        'sentimiento': sent['label'],
        'sentimiento_score': sent['score'],
        'resumen': resumen,
    })

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

,pregunta,respuesta_esperada,respuesta_modelo,qa_score,sentimiento,sentimiento_score,resumen
0,¿Quién creó Python?,Guido van Rossum,Guido van Rossum,0.9682,POS,0.6775,Resume el siguiente texto en una frase: Python...
1,¿En qué año se publicó Python?,1991,1991,0.9627,POS,0.6775,Resume el siguiente texto en una frase: Python...
2,¿Qué empresa introdujo los Transformers?,Google,Google,0.7415,NEU,0.5922,Resume el siguiente texto en una frase: Los Tr...
3,¿En qué año se publicó el paper?,2017,2017,0.9160,NEU,0.5922,Resume el siguiente texto en una frase: Los Tr...
4,¿Qué tipo de arquitectura usa BERT?,encoder-only,encoder-only,0.3046,NEU,0.7327,Resume el siguiente texto en una frase: BERT e...
5,¿Qué predice GPT?,la siguiente palabra,la siguiente palabra,0.6238,NEU,0.7801,Resume el siguiente texto en una frase: GPT pr...
6,¿Qué permite la Self-Attention?,que cada token observe a todos los demás tokens,que cada token observe a todos los demás token...,0.1757,NEU,0.8622,Resume el siguiente texto en una frase: La cap...
7,¿Cómo reformula T5 las tareas?,texto a texto,problemas de texto a texto,0.2303,NEG,0.5978,Resume el siguiente texto en una frase: T5 ref...


In [13]:
df_resultados

,pregunta,respuesta_esperada,respuesta_modelo,qa_score,sentimiento,sentimiento_score,resumen
0,¿Quién creó Python?,Guido van Rossum,Guido van Rossum,0.9682,POS,0.6775,Resume el siguiente texto en una frase: Python...
1,¿En qué año se publicó Python?,1991,1991,0.9627,POS,0.6775,Resume el siguiente texto en una frase: Python...
2,¿Qué empresa introdujo los Transformers?,Google,Google,0.7415,NEU,0.5922,Resume el siguiente texto en una frase: Los Tr...
3,¿En qué año se publicó el paper?,2017,2017,0.9160,NEU,0.5922,Resume el siguiente texto en una frase: Los Tr...
4,¿Qué tipo de arquitectura usa BERT?,encoder-only,encoder-only,0.3046,NEU,0.7327,Resume el siguiente texto en una frase: BERT e...
5,¿Qué predice GPT?,la siguiente palabra,la siguiente palabra,0.6238,NEU,0.7801,Resume el siguiente texto en una frase: GPT pr...
6,¿Qué permite la Self-Attention?,que cada token observe a todos los demás tokens,que cada token observe a todos los demás token...,0.1757,NEU,0.8622,Resume el siguiente texto en una frase: La cap...
7,¿Cómo reformula T5 las tareas?,texto a texto,problemas de texto a texto,0.2303,NEG,0.5978,Resume el siguiente texto en una frase: T5 ref...


## 5) Texto personalizado del usuario

In [14]:
# Escribe tu codigo aqui
texto_ejemplo = "Python es un lenguaje de programación versátil y fácil de aprender, ampliamente utilizado en ciencia de datos, desarrollo web y automatización."

pregunta = "¿Para qué se utiliza Python?"
respuesta = responder_pregunta(texto_ejemplo, pregunta)
print(f"Pregunta: {pregunta}")
print(f"Respuesta: {respuesta['answer']} (score: {respuesta['score']})")

Pregunta: ¿Para qué se utiliza Python?
Respuesta: ciencia de datos, desarrollo web y automatización (score: 0.4926)


## 6) Metricas agregadas del proyecto

In [ ]:
# Escribe tu codigo aqui


## 7) Mejoras propuestas
- Usar modelos multilingues para documentos en espanol.
- Agregar chunking para contextos largos.
- Evaluar con RAG sobre base vectorial corporativa.
- Registrar trazas y scores en un dashboard de monitoreo.

## Resultados
El asistente combino tres capacidades sobre CSV y texto libre, con metricas basicas de exactitud QA y tono del documento.

## Conclusiones
Un sistema documental util une comprension (BERT), extraccion (QA) y sintesis (GPT). La calidad mejora con datos del dominio y evaluacion continua.

## Ejercicios guiados resueltos
**Ejercicio:** Crea funcion `procesar_documento(texto, preguntas)` que devuelva JSON.

**Solucion:**

In [ ]:
# Escribe tu codigo aqui


## Ejercicios propuestos
1. Empaqueta el asistente como API FastAPI.
2. Agrega validacion de entrada y limites de tokens.
3. Disena pruebas unitarias para `responder_pregunta`.

## Preguntas de reflexion
1. Que componente fallaria primero en documentos de 50 paginas?
2. Como combinarias este pipeline con embeddings de Clase 01?
3. Que metricas de negocio reportarias ademas de exact match?